In [1]:
import os
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# --- CORRECCIÓN DE LA RUTA ---
BASE_DIR = r"../../02_Clustering/outputs"

#BASE_DIR = r"C:\Users\artur\OneDrive\Documents\TrabajoTesis\FinalResultsFolder\ProcessedCluster"
INPUT_FILE = os.path.join(BASE_DIR, "metricas_dbcv_regional.csv")

print(f"📊 LEYENDO: {INPUT_FILE}")

if os.path.exists(INPUT_FILE):
    df_final = pd.read_csv(INPUT_FILE)
    
    # --- ASEGURAR NUMÉRICOS PARA PLOTLY ---
    for col in ['Latitud', 'Longitud']:
        if df_final[col].dtype == object:
            df_final[col] = pd.to_numeric(df_final[col].astype(str).str.replace(',', '.'), errors='coerce')
            
    df_final = df_final.dropna(subset=['Latitud', 'Longitud'])
    print(f"✅ Cargadas {len(df_final)} minas válidas.")
else:
    print(f"❌ ERROR: No se encuentra el archivo en la ruta especificada.")
    print(f"   Ruta buscada: {INPUT_FILE}")
    print(f"\n💡 SOLUCIÓN: Actualiza BASE_DIR a la ubicación correcta del archivo:")
    print(f"   BASE_DIR = r'../../02_Clustering/outputs'")
    raise FileNotFoundError(f"No se encuentra el archivo: {INPUT_FILE}")

# ------------------------------
# 2. TABLA DE ESTADÍSTICAS
# ------------------------------
stats = []
valid_df = df_final[~df_final['Cluster_ID'].str.contains('Ruido', na=False, case=False)].copy()

for cid in valid_df['Cluster_ID'].unique():
    sub = valid_df[valid_df['Cluster_ID'] == cid]
    
    # --- CONTAR CATEGORÍAS ---
    cats = sub['CategoriaFaena'].value_counts()
    
    c_a = cats.get('CATEGORIA A', 0)
    c_b = cats.get('CATEGORIA B', 0)
    c_c = cats.get('CATEGORIA C', 0)
    c_d = cats.get('CATEGORIA D', 0)
    
    # --- CÁLCULO DE RATIOS ---
    ab = c_a + c_b
    cd = c_c + c_d
    total_abcd = ab + cd
    
    # Ratio 1: (A+B) / (C+D)
    if cd > 0:
        ratio_1 = round(ab / cd, 2)
    else:
        ratio_1 = "Inf" # O usar 0 o "-" si prefieres
        
    # Ratio 2: (C+D) / (A+B)
    if ab > 0:
        ratio_2 = round(cd / ab, 2)
    else:
        ratio_2 = "Inf"

    # Ratio 3: (C+D) / (A+B+C+D)
    if total_abcd > 0:
        ratio_3 = round(cd / total_abcd, 2)
    else:
        ratio_3 = 0

    # Empresas principales
    empresa_counts = sub['NombreEmpresa'].value_counts().head(3)
    empresas_txt = "<br>".join([f"• {e[:15]}.. ({c})" for e, c in empresa_counts.items()])
    
    stats.append({
        'Valle': cid, 
        # 'Nivel': Eliminado (Tier)
        'Región': ", ".join(sub['Region_Norm'].dropna().unique())[:20],       
        'Total': len(sub),
        'Cat A': c_a,
        'Cat B': c_b,
        'Cat C': c_c,
        'Cat D': c_d,
        '(A+B)/(C+D)': ratio_1,
        '(C+D)/(A+B)': ratio_2,
        '(C+D)/Tot': ratio_3,
        'Empresas': empresas_txt
    })

df_stats = pd.DataFrame(stats).sort_values('Total', ascending=False)

if not df_stats.empty:
    # --- TABLA ---
    # Definimos las columnas y sus anchos
    # Valle, Región, Total, A, B, C, D, R1, R2, R3, Empresas
    col_widths = [100, 80, 50, 40, 40, 40, 40, 60, 60, 60, 150]
    headers = ['ID Valle', 'Región', 'Minas', 'A', 'B', 'C', 'D', 
               '(A+B)/(C+D)', '(C+D)/(A+B)', '(C+D)/Tot', 'Dueños']
    
    # Extraemos los valores en orden
    cell_values = [
        df_stats['Valle'], 
        df_stats['Región'], 
        df_stats['Total'],
        df_stats['Cat A'], 
        df_stats['Cat B'], 
        df_stats['Cat C'], 
        df_stats['Cat D'],
        df_stats['(A+B)/(C+D)'],
        df_stats['(C+D)/(A+B)'],
        df_stats['(C+D)/Tot'],
        df_stats['Empresas']
    ]

    fig_table = go.Figure(data=[go.Table(
        columnwidth=col_widths,
        header=dict(values=headers,
                    fill_color='#2c3e50', font=dict(color='white'), align='left'),
        cells=dict(values=cell_values,
                   fill_color='#f8f9fa', align='left')
    )])
    fig_table.update_layout(title="Resumen de Valles Mineros (Por Categoría y Ratios)", 
                            height=500, margin=dict(l=5,r=5,t=40,b=5))
    fig_table.show()

    # --- MAPA ---
    # Eliminamos 'Tier' de hover_data si existía
    hover_cols = ["NombreEmpresa"]

    
    fig_map = px.scatter_mapbox(
        valid_df,
        lat="Latitud",
        lon="Longitud",
        color="Cluster_ID",
        hover_name="NombreFaena",
        hover_data=hover_cols, # Se quitó 'Tier'
        zoom=4,
        height=800,
        title="Distribución Geográfica de Valles Mineros Detectados",
        mapbox_style="open-street-map"
    )
    
    fig_map.update_traces(marker=dict(size=6, opacity=0.8)) 
    
    fig_map.show()
else:
    print("⚠️ No se encontraron clusters válidos para visualizar.")

📊 LEYENDO: ../../02_Clustering/outputs/metricas_dbcv_regional.csv
✅ Cargadas 7928 minas válidas.


/var/folders/8g/vqdnj8vx7l98mbf5k83d7n1m0000gn/T/ipykernel_80706/3114632431.py:131: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

